# B7. Mô hình nền và kiểm định chéo

Huấn luyện ba mô hình, **hồi quy logistic → rừng ngẫu nhiên → LightGBM**, với tham số mặc định, chấm bằng kiểm định chéo 5 fold phân tầng
**chỉ trên tập train** (246.008 hồ sơ). Tập test chưa được chạm. Mỗi mô hình chạy hai lần: không trọng số lớp và trọng số cân bằng (`class_weight="balanced"`).
Đầu ra: AUC kiểm định chéo của từng mô hình; quyết định có dùng trọng số lớp ở B8 không; `ml/data/folds.csv` (fold của từng hồ sơ train, B8–B10 dùng lại).

## 0. Nguồn đã tra và cách áp dụng

| Nguồn | Nội dung | Áp dụng |
|---|---|---|
| scikit-learn, *Common pitfalls and recommended practices* (mục Data leakage) | Không `fit` bộ tiền xử lý trên dữ liệu dùng để chấm; kể cả `SimpleImputer`, `StandardScaler`. Gói tiền xử lý và mô hình vào `Pipeline` rồi đưa vào `cross_validate` thì mỗi fold chỉ học từ phần huấn luyện của fold đó | Mọi bước điền thiếu, mã hóa, chuẩn hóa nằm trong pipeline (mục 2) |
| scikit-learn 1.4, *Release notes*; tài liệu `RandomForestClassifier` (bản 1.9 đang dùng) | Rừng ngẫu nhiên hỗ trợ ô trống trực tiếp: ở mỗi lần tách, cây học ô trống đi nhánh trái hay phải theo mức giảm gini | Rừng ngẫu nhiên không điền thiếu, giống LightGBM |
| LightGBM, *Advanced Topics* (bản 4.7) | Ô trống xử lý mặc định; cột phân loại mã hóa số nguyên, tách theo Fisher (1958), thường tốt hơn one-hot | LightGBM nhận cột chữ dạng mã số nguyên + `categorical_feature` |
| scikit-learn, tài liệu `LogisticRegression` | `lbfgs` là bộ giải mặc định; hội tụ cần các cột cùng thang đo; mặc định điều chuẩn L2, `C` = 1 | Hồi quy logistic có chuẩn hóa; điều chuẩn L2 mặc định (B6: 9 cột VIF > 10) |
| Josse và cộng sự (2019), *On the consistency of supervised learning with missing values*, arXiv:1902.06931 | Điền hằng số (trung bình) trước khi học vẫn cho dự đoán nhất quán khi thiếu không mang thông tin; khi thiếu mang thông tin cần cách để mô hình thấy được chỗ thiếu | Hồi quy logistic: điền trung vị **kèm cột cờ thiếu** (B5: thiếu không ngẫu nhiên, không có CIC 10,12% so với 7,73%) |
| Train in Data, *Class weights do not improve model performance*; nghiên cứu CT-DEB26 (arXiv:2606.30236) | Trọng số lớp gần như không đổi ROC-AUC (chênh nằm trong sai số ước lượng); nó dời xác suất, ảnh hưởng metric phụ thuộc ngưỡng | Đo cả hai cách trên cùng fold, quyết định bằng luật đặt trước (mục 3). Không SMOTE (bài học bản cũ) |
| Lessmann, Baesens, Seow, Thomas (2015), *Benchmarking state-of-the-art classification algorithms for credit scoring: An update of research*, EJOR 247(1) | Hồi quy logistic là mốc so sánh chuẩn trong chấm điểm tín dụng; mô hình tổ hợp cây thường vượt | Thứ tự LR (mốc) → RF → LightGBM |

Không áp dụng ở bước này: tinh chỉnh siêu tham số, dừng sớm (B8); hiệu chỉnh xác suất (B9); ngưỡng (B10).

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.compose import make_column_transformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
SEED = 42
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

f = pd.read_parquet(ROOT / "ml" / "data" / "features.parquet")
tap = pd.read_csv(ROOT / "ml" / "data" / "split.csv")
f = f[f.SK_ID_CURR.isin(tap.SK_ID_CURR[tap.tap == "train"])].reset_index(drop=True)
y = f.pop("TARGET")
X = f.drop(columns="SK_ID_CURR")
CHU = X.select_dtypes(exclude="number").columns.tolist()
SO = [c for c in X.columns if c not in CHU]
print(f"Train: {len(X):,} hồ sơ, TARGET=1 {y.mean():.4%}; {len(SO)} cột số, {len(CHU)} cột chữ")

Train: 246,008 hồ sơ, TARGET=1 8.0729%; 109 cột số, 15 cột chữ


## 1. Chia fold

5 fold **phân tầng theo TARGET** (mỗi fold giữ tỷ lệ 8,07%), xáo trộn với seed 42. Mỗi vòng huấn luyện trên 4 fold (≈ 196.800 hồ sơ), chấm trên fold còn lại (≈ 49.200).
Lưu số fold của từng hồ sơ để B8–B10 dùng đúng các fold này (so sánh giữa các bước là so trên cùng cách chia).

In [2]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold = np.empty(len(X), dtype=int)
for k, (_, va) in enumerate(cv.split(X, y)):
    fold[va] = k
pd.DataFrame({"SK_ID_CURR": f.SK_ID_CURR, "fold": fold}).to_csv(ROOT / "ml" / "data" / "folds.csv", index=False)
pd.DataFrame({"so_ho_so": pd.Series(fold).value_counts().sort_index(), "ty_le_TARGET": y.groupby(fold).mean()}).rename_axis("fold")

,so_ho_so,ty_le_TARGET
fold,,
0,49202,0.0807
1,49202,0.0807
2,49202,0.0807
3,49201,0.0807
4,49201,0.0807


## 2. Tiền xử lý trong pipeline

Mỗi mô hình là một `Pipeline`; `cross_validate` gọi `fit` của cả pipeline trên 4 fold huấn luyện, nên trung vị để điền, danh sách nhóm của cột chữ,
trung bình và độ lệch chuẩn để chuẩn hóa đều **học lại ở từng vòng**, không thấy fold đang chấm.

| Mô hình | Cột số | Cột chữ | Lý do |
|---|---|---|---|
| Hồi quy logistic | Điền trung vị + cột cờ thiếu; chuẩn hóa | Ô trống thành nhóm "THIEU"; one-hot, nhóm không gặp lúc huấn luyện bỏ qua | Không nhận ô trống; hệ số phụ thuộc thang đo; cờ thiếu giữ thông tin "không có CIC" |
| Rừng ngẫu nhiên | Giữ nguyên, kể cả ô trống | One-hot như trên | Cây tự học ô trống đi nhánh nào; mã số nguyên sẽ áp thứ tự giả lên nhóm |
| LightGBM | Giữ nguyên, kể cả ô trống | Mã số nguyên (ô trống, nhóm lạ → trống), khai báo `categorical_feature` | Tách nhóm trực tiếp theo tài liệu LightGBM |

Không biến đổi log, không chặn giá trị lớn (bài học bản cũ: không xử lý ngoại lai). Tham số mô hình để **mặc định** (trừ `max_iter` của LR để hội tụ, `random_state`, `n_jobs`);
tinh chỉnh là việc của B8.

In [3]:
onehot = make_pipeline(SimpleImputer(strategy="constant", fill_value="THIEU"), OneHotEncoder(handle_unknown="ignore"))


def mo_hinh(ten, trong_so):
    if ten == "LR":
        tien_xu_ly = make_column_transformer(
            (make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), SO), (onehot, CHU))
        return make_pipeline(tien_xu_ly, LogisticRegression(max_iter=3000, class_weight=trong_so))
    if ten == "RF":
        tien_xu_ly = make_column_transformer((onehot, CHU), remainder="passthrough")
        return make_pipeline(tien_xu_ly, RandomForestClassifier(class_weight=trong_so, random_state=SEED, n_jobs=-1))
    tien_xu_ly = make_column_transformer(
        (OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan, encoded_missing_value=np.nan), CHU),
        remainder="passthrough", verbose_feature_names_out=False).set_output(transform="pandas")
    return make_pipeline(tien_xu_ly, LGBMClassifier(class_weight=trong_so, random_state=SEED, n_jobs=-1, verbose=-1))


for ten in ["LR", "RF", "LGBM"]:
    p = mo_hinh(ten, None)
    Xt = p[:-1].fit_transform(X.head(20000))
    print(f"{ten}: {X.shape[1]} cột vào → {Xt.shape[1]} cột vào mô hình")

LR: 124 cột vào → 319 cột vào mô hình
RF: 124 cột vào → 248 cột vào mô hình
LGBM: 124 cột vào → 124 cột vào mô hình


## 3. Luật chọn trọng số lớp (đặt trước khi chạy)

`class_weight="balanced"` nhân trọng số mỗi hồ sơ TARGET=1 lên ≈ 0,5 / 0,0807 ≈ 6,2 lần và TARGET=0 xuống ≈ 0,54 lần, để hai lớp đóng góp ngang nhau vào hàm mất mát.
Cách này dời xác suất lên (trung bình dự đoán ≈ 50% thay vì 8%), nên B9 phải hiệu chỉnh lại; cái cần biết là nó có làm **thứ hạng** (AUC) tốt hơn không.

**Luật:** với mỗi mô hình, so AUC kiểm định của hai cách trên **cùng 5 fold**. Chỉ giữ trọng số cân bằng nếu AUC tăng **ở cả 5 fold**;
nếu không thì dùng không trọng số (đơn giản hơn, xác suất không bị dời). Nếu trọng số không có tác dụng thì mỗi fold tăng hay giảm như tung đồng xu,
xác suất cả 5 fold cùng tăng là 1/2⁵ ≈ 3%, nên "tăng ở cả 5 fold" là bằng chứng đủ chặt mà không cần kiểm định nào ngoài phạm vi.

In [4]:
kq, theo_fold = [], {}
for ten in ["LR", "RF", "LGBM"]:
    for trong_so in [None, "balanced"]:
        t = time.time()
        r = cross_validate(mo_hinh(ten, trong_so), X, y, cv=cv, scoring=["roc_auc", "average_precision"], return_train_score=True,
                           n_jobs=5 if ten == "LR" else 1,
                           params={"lgbmclassifier__categorical_feature": CHU} if ten == "LGBM" else None)
        nhan = (ten, trong_so or "không")
        theo_fold[nhan] = r["test_roc_auc"]
        kq.append({"mô hình": ten, "trọng số": trong_so or "không",
                   "AUC kiểm định": r["test_roc_auc"].mean(), "độ lệch chuẩn": r["test_roc_auc"].std(),
                   "PR-AUC kiểm định": r["test_average_precision"].mean(), "AUC huấn luyện": r["train_roc_auc"].mean(),
                   "phút": (time.time() - t) / 60})
        print(nhan, np.round(r["test_roc_auc"], 4))
kq = pd.DataFrame(kq).set_index(["mô hình", "trọng số"])
kq

('LR', 'không') [0.7664 0.767  0.7667 0.7651 0.765 ]


('LR', 'balanced') [0.7664 0.7665 0.7673 0.7652 0.7655]


('RF', 'không') [0.7372 0.7387 0.7339 0.7357 0.7324]


('RF', 'balanced') [0.7556 0.758  0.754  0.7543 0.7555]


('LGBM', 'không') [0.7794 0.7786 0.7785 0.7755 0.7777]


('LGBM', 'balanced') [0.7775 0.7804 0.7799 0.7758 0.7777]


AUC kiểm định  độ lệch chuẩn  PR-AUC kiểm định  \
mô hình trọng số                                                   
LR      không            0.7660         0.0008            0.2498   
        balanced         0.7662         0.0008            0.2471   
RF      không            0.7356         0.0023            0.2258   
        balanced         0.7555         0.0014            0.2287   
LGBM    không            0.7779         0.0013            0.2652   
        balanced         0.7783         0.0017            0.2660   

                  AUC huấn luyện   phút  
mô hình trọng số                         
LR      không             0.7701 0.4599  
        balanced          0.7712 0.7941  
RF      không             1.0000 1.2401  
        balanced          1.0000 0.8586  
LGBM    không             0.8379 0.2733  
        balanced          0.8378 0.2676

## 4. Trọng số lớp: áp luật mục 3

In [5]:
chenh = pd.DataFrame({ten: theo_fold[(ten, "balanced")] - theo_fold[(ten, "không")] for ten in ["LR", "RF", "LGBM"]}).rename_axis("fold")
print("Chênh AUC (cân bằng − không trọng số) từng fold:")
display(chenh)
chon = {ten: "balanced" if (chenh[ten] > 0).all() else "không" for ten in chenh}
print("Số fold tăng:", (chenh > 0).sum().to_dict())
print("Chọn cho B8:", chon)

Chênh AUC (cân bằng − không trọng số) từng fold:


,LR,RF,LGBM
fold,,,
0,0.0000,0.0184,-0.0019
1,-0.0005,0.0193,0.0019
2,0.0006,0.0202,0.0014
3,0.0001,0.0186,0.0003
4,0.0005,0.0231,-0.0000


Số fold tăng: {'LR': 4, 'RF': 5, 'LGBM': 3}
Chọn cho B8: {'LR': 'không', 'RF': 'balanced', 'LGBM': 'không'}


## 5. Mô hình nền theo cấu hình đã chọn

AUC huấn luyện so với AUC kiểm định cho thấy mức học thuộc: chênh lớn nghĩa là mô hình nhớ dữ liệu huấn luyện, B8 cần điều chuẩn mạnh hơn.

In [6]:
nen = kq.loc[[(ten, chon[ten]) for ten in ["LR", "RF", "LGBM"]]].copy()
nen["chênh huấn luyện − kiểm định"] = nen["AUC huấn luyện"] - nen["AUC kiểm định"]
nen


,,AUC kiểm định,độ lệch chuẩn,PR-AUC kiểm định,AUC huấn luyện,phút,chênh huấn luyện − kiểm định
mô hình,trọng số,,,,,,
LR,không,0.7660,0.0008,0.2498,0.7701,0.4599,0.0041
RF,balanced,0.7555,0.0014,0.2287,1.0000,0.8586,0.2445
LGBM,không,0.7779,0.0013,0.2652,0.8379,0.2733,0.0600


In [7]:
fold_nen = pd.DataFrame({ten: theo_fold[(ten, chon[ten])] for ten in ["LR", "RF", "LGBM"]}).rename_axis("fold")
fold_nen["RF − LR"] = fold_nen.RF - fold_nen.LR
fold_nen["LGBM − RF"] = fold_nen.LGBM - fold_nen.RF
fold_nen

,LR,RF,LGBM,RF − LR,LGBM − RF
fold,,,,,
0,0.7664,0.7556,0.7794,-0.0108,0.0238
1,0.7670,0.7580,0.7786,-0.0090,0.0206
2,0.7667,0.7540,0.7785,-0.0127,0.0245
3,0.7651,0.7543,0.7755,-0.0108,0.0212
4,0.7650,0.7555,0.7777,-0.0095,0.0222


## 6. Nhận xét (sau thực nghiệm)

- **Trọng số lớp:** LR tăng 4/5 fold (chênh trung bình +0,0002), LightGBM 3/5 (+0,0003): không đạt luật, dùng không trọng số. RF tăng **cả 5 fold** (+0,018 đến +0,023): dùng `balanced` ở B8.
  Giải thích (chưa kiểm chứng): RF mặc định mọc cây đến khi lá thuần (`min_samples_leaf` = 1); không trọng số thì gini gần như chỉ do lớp 0 quyết định nên các lần tách đầu ít tách hồ sơ TARGET=1;
  có trọng số thì hai lớp góp ngang nhau vào gini (weighted random forest của Chen, Liaw, Breiman, 2004). Khi B8 tăng kích thước lá, mức chênh này có thể khác.
- **Thứ tự mô hình nền:** LightGBM 0,778 > LR 0,766 > RF 0,756 ở **cả 5 fold**. RF mặc định đứng sau LR vì học thuộc: AUC huấn luyện 1,000 so với kiểm định 0,756;
  LightGBM chênh 0,060; LR 0,004. Thứ tự này là của tham số mặc định, chưa phải kết luận so sánh (B8 tinh chỉnh, B11 chấm trên test).
- Độ lệch chuẩn giữa các fold 0,001–0,002, nhỏ hơn nhiều các chênh giữa mô hình (0,010–0,024).